# Lesson 2.3 — PDF 表格抽取


**目标**：从 PDF 中抽取表格，保存为 CSV，并转为 Markdown 供 LLM 使用。

---

对比两种工具：

| 工具 | 特点 | 适用场景 |
|------|------|----------|
| **pdfplumber** | 内置 `extract_tables()`，上手快 | 一般表格 |
| **camelot** | 专业表格库，分两种模式 | 关键报表 / 财务数据 |

### camelot 两种模式

| 模式 | 适用 | 原理 |
|------|------|------|
| **lattice** | 表格有清晰边框线 | 检测线段交点找单元格 |
| **stream** | 表格无线，靠空白对齐 | 检测文本块的对齐 |

**经验**：先试 `lattice`，失败回退 `stream`。

## 表格：PDF 解析的「地狱难度」

### 为什么表格难？
1. 表格在 PDF 里就是**一堆线 + 一堆文本块**，没有「行」「列」的概念
2. 单元格合并、跨页表格、嵌套表格...
3. **同一份 PDF 里可能有多种表格风格**

### Lattice（有线）vs Stream（无线）

| 模式 | 原理 | 适用 | 局限 |
|------|------|------|------|
| **Lattice** | 检测线段交点框出单元格 | 有清晰边框 | 无边框完全识别不出 |
| **Stream** | 检测文本块对齐，用空白推断列 | 无边框也能用 | 列对齐不严时会错 |

> ⚠️ **陷阱 #6**：表格当文本切 → 行列对应丢失

### 本节处理逻辑图

```mermaid
flowchart LR
    A[PDF 文件] --> B[逐页遍历]

    B --> C[pdfplumber<br/>extract_tables]
    B --> D[camelot lattice<br/>有边框]
    B --> E[camelot stream<br/>无边框回退]

    C --> F[DataFrame<br/>首行作表头]
    D --> F
    E --> F

    F --> G[保存 CSV<br/>pdfplumber/ · camelot/]
    G --> H[table_to_markdown<br/>df.to_markdown]
    H --> I[保存 .md<br/>markdown/]
    I --> J[table chunk 入库<br/>type=table · page]

    style C fill:#e8f4fd
    style D fill:#e8f4fd
    style E fill:#e8f4fd
    style I fill:#fff3cd
    style J fill:#d4edda
```

> ⚠️ 表格单独成 chunk，不要当普通文本切分，否则行列对应丢失。

## 0. 安装依赖

In [34]:
%pip install pdfplumber pandas tabulate -q
# camelot 可选（依赖 OpenCV，安装较重）
# %pip install "camelot-py[cv]" -q

Note: you may need to restart the kernel to use updated packages.


In [35]:
from pathlib import Path

import pandas as pd
import pdfplumber

PDF_PATH = Path("doc/attention_is_all_you_need.pdf")
OUTPUT_DIR = Path("out")

if not PDF_PATH.exists():
    raise FileNotFoundError(f"请把 PDF 放到: {PDF_PATH.resolve()}")

In [36]:
print(OUTPUT_DIR)

out


## 1. 方法 A — pdfplumber 抽表

`page.extract_tables()` 返回二维列表，第一行通常是表头。

In [37]:
def extract_with_pdfplumber(pdf_path: Path, output_dir: Path):
    """pdfplumber 通用版"""
    output_dir.mkdir(parents=True, exist_ok=True)
    count = 0

    with pdfplumber.open(pdf_path) as pdf:
        for page_num, page in enumerate(pdf.pages, 1):
            tables = page.extract_tables()
            for idx, table in enumerate(tables):
                if not table or len(table) < 2:
                    continue

                df = pd.DataFrame(table[1:], columns=table[0])
                csv_path = output_dir / f"plumber_p{page_num}_t{idx}.csv"
                df.to_csv(csv_path, index=False)
                count += 1
                print(f"✅ pdfplumber 第 {page_num} 页表格 {idx} → {csv_path.name} ({len(df)} 行)")

    print(f"\n📊 pdfplumber 共抽到 {count} 张表")
    return count

In [38]:
table_dir = OUTPUT_DIR / "pdfplumber"

plumber_count = extract_with_pdfplumber(PDF_PATH, table_dir)

✅ pdfplumber 第 9 页表格 0 → plumber_p9_t0.csv (7 行)
✅ pdfplumber 第 10 页表格 0 → plumber_p10_t0.csv (5 行)
✅ pdfplumber 第 13 页表格 0 → plumber_p13_t0.csv (8 行)
✅ pdfplumber 第 13 页表格 2 → plumber_p13_t2.csv (7 行)
✅ pdfplumber 第 14 页表格 0 → plumber_p14_t0.csv (10 行)
✅ pdfplumber 第 14 页表格 1 → plumber_p14_t1.csv (2 行)
✅ pdfplumber 第 15 页表格 0 → plumber_p15_t0.csv (3 行)
✅ pdfplumber 第 15 页表格 1 → plumber_p15_t1.csv (5 行)

📊 pdfplumber 共抽到 8 张表


### 1.1 预览：单页有多少张表？

先不保存，直接看 `extract_tables()` 的原始结构。

In [39]:
with pdfplumber.open(PDF_PATH) as pdf:
    for page_num, page in enumerate(pdf.pages, 1):
        tables = page.extract_tables()
        if tables:
            print(f"第 {page_num} 页: {len(tables)} 张表")
            for idx, table in enumerate(tables):
                print(f"  表 {idx}: {len(table)} 行 × {len(table[0]) if table else 0} 列")
                print(f"  表头: {table[0]}")

第 9 页: 1 张表
  表 0: 8 行 × 1 列
  表头: ['train\nN d d h d d P ϵ\nmodel ff k v drop ls steps']
第 10 页: 1 张表
  表 0: 6 行 × 1 列
  表头: ['Training']
第 13 页: 3 张表
  表 0: 9 行 × 39 列
  表头: [None, '', None, '', None, '', None, '', None, '', None, '', None, '', None, '', None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None]
  表 1: 1 行 × 2 列
  表头: ['', '']
  表 2: 8 行 × 33 列
  表头: ['tI', 'si', 'ni', 'si', 'tir', 'ta', 'a', 'yt', 'fo', 'n', 'st', 'e', 'd', 'w', 's', 'e', '9', 'g', 'e', 'n', 'ro', 'g', 'ss', 'er', 'tlu', '.', '>', '>', '>', '>', '>', '>', '>']
第 14 页: 2 张表
  表 0: 11 行 × 59 列
  表头: [None, '', None, '', None, None, '', None, None, '', None, '', None, None, '', None, None, '', None, '', None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, Non

## camelot 实战

```python
import camelot

tables = camelot.read_pdf("file.pdf", pages="all", flavor="lattice")
for i, t in enumerate(tables):
    print(t.parsing_report)  # accuracy, whitespace
    t.df.to_csv(f"table_{i}.csv")

tables = camelot.read_pdf("file.pdf", pages="all", flavor="stream")

def best_extract(pdf):
    lat = camelot.read_pdf(pdf, flavor="lattice")
    st = camelot.read_pdf(pdf, flavor="stream")
    return lat if avg_acc(lat) > avg_acc(st) else st
```

每张表 `parsing_report['accuracy']` < 60 的结果通常不可靠，应丢弃。


## 2. 方法 B — camelot 双模式

camelot 会给出 `parsing_report["accuracy"]`，低于 60 的结果通常不可靠，应丢弃。

In [40]:
%pip install camelot-py[cv]

Note: you may need to restart the kernel to use updated packages.


In [41]:
def extract_with_camelot(pdf_path: Path, output_dir: Path):
    """camelot 双模式 + 选最佳"""
    try:
        import camelot
    except ImportError:
        print("⚠️ camelot 未安装，跳过。安装：pip install camelot-py[cv]")
        return 0

    output_dir.mkdir(parents=True, exist_ok=True)
    count = 0

    print("🔍 尝试 lattice 模式（有边框）...")
    try:
        tables_lattice = camelot.read_pdf(str(pdf_path), pages="all", flavor="lattice")
        print(f"  找到 {len(tables_lattice)} 张表")
        for i, t in enumerate(tables_lattice):
            score = t.parsing_report["accuracy"]
            if score < 60:
                continue
            csv_path = output_dir / f"camelot_lattice_{i}.csv"
            t.df.to_csv(csv_path, index=False)
            count += 1
            print(f"  ✅ lattice 表格 {i} → {csv_path.name} (accuracy={score:.1f})")
    except Exception as e:
        print(f"  ⚠️ lattice 失败: {e}")

    print("\n🔍 尝试 stream 模式（无边框）...")
    try:
        tables_stream = camelot.read_pdf(str(pdf_path), pages="all", flavor="stream")
        print(f"  找到 {len(tables_stream)} 张表")
        for i, t in enumerate(tables_stream):
            score = t.parsing_report["accuracy"]
            if score < 60:
                continue
            csv_path = output_dir / f"camelot_stream_{i}.csv"
            t.df.to_csv(csv_path, index=False)
            count += 1
            print(f"  ✅ stream 表格 {i} → {csv_path.name} (accuracy={score:.1f})")
    except Exception as e:
        print(f"  ⚠️ stream 失败: {e}")

    return count

## 表格转 Markdown 喂给 LLM

LLM 读 CSV 很烂，读 Markdown 表格很好。

```python
df = pd.read_csv("table.csv")
md = df.to_markdown(index=False)
```

输出示例：
```markdown
| 产品 | 销量 | 增长率 |
|------|------|--------|
| A    | 100  | +5%    |
| B    | 200  | -3%    |
```

> 💡 **把表格单独成 chunk**，加 metadata：`{"type": "table", "page": 3}`

> ⚠️ **陷阱 #6**：表格当文本切 → 行列对应丢失


## 3. 表格 → Markdown（喂给 LLM）

RAG 场景下，表格最好转成 Markdown 格式，LLM 更容易理解行列关系。

In [42]:
def table_to_markdown(csv_path: Path) -> str:
    """把 CSV 表格转 Markdown，方便塞进 LLM prompt"""
    df = pd.read_csv(csv_path)
    return df.to_markdown(index=False)

In [43]:
csvs = sorted(table_dir.glob("*.csv"))
md_dir = OUTPUT_DIR / "markdown"
md_dir.mkdir(parents=True, exist_ok=True)

print(f"输出目录: {md_dir.resolve()}")
print(f"共 {len(csvs)} 个 CSV 文件\n")

if csvs:
    for csv_path in csvs:
        md_path = md_dir / f"{csv_path.stem}.md"
        md_path.write_text(table_to_markdown(csv_path), encoding="utf-8")
        print(f"✅ {csv_path.name} → {md_path.name}")

    print(f"\n📝 第一张表 ({csvs[0].name}) 预览：")
    print(table_to_markdown(csvs[0]))
else:
    print("未抽到表格。换一份含表格的 PDF 试试。")

输出目录: D:\workspace\doc\面试狂魔\人工智能面试题\AI_coding_interview\02-RAG\05_PDF_process\out\markdown
共 8 个 CSV 文件

✅ plumber_p10_t0.csv → plumber_p10_t0.md
✅ plumber_p13_t0.csv → plumber_p13_t0.md
✅ plumber_p13_t2.csv → plumber_p13_t2.md
✅ plumber_p14_t0.csv → plumber_p14_t0.md
✅ plumber_p14_t1.csv → plumber_p14_t1.md
✅ plumber_p15_t0.csv → plumber_p15_t0.md
✅ plumber_p15_t1.csv → plumber_p15_t1.md
✅ plumber_p9_t0.csv → plumber_p9_t0.md

📝 第一张表 (plumber_p10_t0.csv) 预览：
| Training               |
|:-----------------------|
| WSJonly,discriminative |
| WSJonly,discriminative |
| WSJonly,discriminative |
| WSJonly,discriminative |
| WSJonly,discriminative |
| semi-supervised        |
| semi-supervised        |
| semi-supervised        |
| semi-supervised        |
| semi-supervised        |
| multi-task             |
| generative             |


## 4. 小结

| 场景 | 推荐 |
|------|------|
| 快速验证 | pdfplumber |
| 有边框的报表 | camelot lattice |
| 无边框对齐表 | camelot stream |
| 入库 RAG | CSV → Markdown，表格单独成 chunk |

**课堂练习**：同一份 PDF 对比 pdfplumber 和 camelot 的结果差异。

下一步 → [05_full_pipeline.ipynb](05_full_pipeline.ipynb) 看表格如何单独成 chunk。